In [1]:
import os

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import haversine_distances
from math import radians

from xarray.util.generate_ops import inplace

In [2]:
def make_dataframe_cols_lowercase(df):
    for col in df.columns:
        df.rename(columns={col: col.lower()}, inplace=True)

def add_formatted_join_column(df, join_col_name, drop_cols=None):
    df['sample date (date)'] = pd.to_datetime(df['sample date'], format='%d-%m-%Y')
    df['sample date'] = pd.to_datetime(df['sample date'], format='%d-%m-%Y').dt.strftime('%d-%m-%Y')
    df['latitude'] = df['latitude'].astype(float).round(6)
    df['longitude'] = df['longitude'].astype(float).round(6)
    df[join_col_name] = df['latitude'].astype(str) + "~" + df['longitude'].astype(str) + "~" + df['sample date']

    if drop_cols:
        df.drop(columns=drop_cols, inplace=True)

In [35]:
DATE_PLUS_MINUS_DAYS = 30
PRIMARY_COLUMNS = ['latitude', 'longitude', 'sample date']
DATA_COLUMNS = list({'ca_diss_water', 'ca_diss_water_dl', 'cl_diss_water',
       'cl_diss_water_dl', 'dms_tot_water', 'dms_tot_water_dl',
       'ec_phys_water', 'ec_phys_water_dl', 'f_diss_water', 'f_diss_water_dl',
       'kjel_n_tot_water', 'kjel_n_tot_water_dl', 'k_diss_water',
       'k_diss_water_dl', 'mg_diss_water', 'mg_diss_water_dl',
       'nh4_n_diss_water', 'nh4_n_diss_water_dl', 'no3_no2_n_calc_water_dl',
       'no3_no2_n_diss_water', 'no3_no2_n_diss_water_dl', 'na_diss_water',
       'na_diss_water_dl', 'po4_p_diss_water', 'po4_p_diss_water_dl',
       'p_tot_water', 'p_tot_water_dl', 'so4_diss_water',
       'so4_diss_water_dl', 'si_diss_water', 'si_diss_water_dl', 'tal_diss_water', 'tal_diss_water_dl',  'ph_diss_water', 'ph_diss_water_dl'})

In [4]:
wq_df = pd.read_csv("../../../water_quality_training_dataset.csv")
make_dataframe_cols_lowercase(wq_df)
add_formatted_join_column(wq_df, "join_column")
wq_df['sample date plus'] = wq_df['sample date (date)'] + pd.Timedelta(days=DATE_PLUS_MINUS_DAYS)
wq_df['sample date minus'] = wq_df['sample date (date)'] - pd.Timedelta(days=DATE_PLUS_MINUS_DAYS)

In [5]:
sa_wq_df = None
all_dfs = []
all_columns_set = set()

for region_name in os.listdir("extracted_data"):
    if not region_name.endswith(".csv"):
        print(region_name)
        for filename in os.listdir(f"extracted_data/{region_name}"):
            if filename.endswith(".csv"):
                print(f"\t\t{filename}")
                extracted_df = pd.read_csv(f"extracted_data/{region_name}/{filename}")
                all_columns_set.update(extracted_df.columns)
                all_dfs.append(extracted_df)


Breë
		bore.csv
		no_bore.csv
Buffels et al
		bore.csv
Bushmans et al
		bore.csv
		no_bore.csv
Crocodile (E)
		bore.csv
		no_bore.csv
Gamtoos
		bore.csv
		no_bore.csv
Gourits
		bore.csv
		no_bore.csv
Great Berg et al
		bore.csv
		no_bore.csv
Great Fish
		bore.csv
		no_bore.csv
Great Fish +
Great Kei
		bore.csv
		no_bore.csv
Keiskamma et al
		bore.csv
		no_bore.csv
Kromme et al
		bore.csv
		no_bore.csv
Limpopo
		bore.csv
		no_bore.csv
Limpopo +
Mzimvubu et al
		bore.csv
		no_bore.csv
Olifants (E)
		bore.csv
		no_bore.csv
Olifants (W) et al
		bore.csv
		no_bore.csv
Orange
		bore.csv
		no_bore.csv
Outside borders
Phongolo et al
		bore.csv
		no_bore.csv
Sundays
		bore.csv
		no_bore.csv
Swartkops
		bore.csv
		no_bore.csv
Thukela
		bore.csv
		no_bore.csv
uMngeni et al
		bore.csv
		no_bore.csv
Vaal
		bore.csv
		no_bore.csv


In [6]:
all_dfs[0]

,mon_feature_id,date_time,sample_begin_depth,institution_abbr,preservative_abbr,Ca_Diss_Water,Ca_Diss_Water_dl,Cl_Diss_Water,Cl_Diss_Water_dl,DMS_Tot_Water,...,data_desc,data_num_records,data_first_date,data_last_date,data_med_ec,data_flow,data_lat,data_lng,region_name,bore_type
0,200189778,2015-03-31 12:45:00,0,DWS-RQIS,HGCL2,8.678,2.5,18.699,1,69.383,...,Goudini Spa Replacement Borehole BE00120 (N_GW),6,2015-03-31,2017-10-20,11,200189778,-33.66944,19.26533,Breë,bore
1,200189778,2015-09-23 11:55:00,#n/a,DWS-RQIS,HGCL2,9.122,2.5,14.544,2,54.177,...,Goudini Spa Replacement Borehole BE00120 (N_GW),6,2015-03-31,2017-10-20,11,200189778,-33.66944,19.26533,Breë,bore
2,200189778,2016-04-13 14:00:00,#n/a,DWS-RQIS,HGCL2,9.5,2.5,16.6,2,63.972,...,Goudini Spa Replacement Borehole BE00120 (N_GW),6,2015-03-31,2017-10-20,11,200189778,-33.66944,19.26533,Breë,bore
3,200189778,2016-09-16 13:01:00,0,DWS-RQIS,HGCL2,8.5,2.5,15.3,2,62.987,...,Goudini Spa Replacement Borehole BE00120 (N_GW),6,2015-03-31,2017-10-20,11,200189778,-33.66944,19.26533,Breë,bore
4,200189778,2017-04-25 13:56:00,#n/a,DWS-RQIS,HGCL2,8.4,2.5,17.1,2,70.522,...,Goudini Spa Replacement Borehole BE00120 (N_GW),6,2015-03-31,2017-10-20,11,200189778,-33.66944,19.26533,Breë,bore
5,200189778,2017-10-20 11:20:00,#n/a,DWS-RQIS,HGCL2,9.4,2.5,20.4,2,81.129,...,Goudini Spa Replacement Borehole BE00120 (N_GW),6,2015-03-31,2017-10-20,11,200189778,-33.66944,19.26533,Breë,bore
6,200189778,2021-10-19 13:14:00,#n/a,WC-CSIR-STELLENBOSCH,NONE,8.5,1,17,2,63.299,...,Goudini Spa Replacement Borehole BE00120 (N_GW),6,2015-03-31,2017-10-20,11,200189778,-33.66944,19.26533,Breë,bore
7,200189778,2022-10-18 13:54:00,0,DWS-RQIS,NONE,5.006,0.13,17.384,3,53.162,...,Goudini Spa Replacement Borehole BE00120 (N_GW),6,2015-03-31,2017-10-20,11,200189778,-33.66944,19.26533,Breë,bore
8,200189778,2023-05-09 13:26:00,0,DWS-RQIS,NONE,#n/a,#n/a,18.724,3,64.085,...,Goudini Spa Replacement Borehole BE00120 (N_GW),6,2015-03-31,2017-10-20,11,200189778,-33.66944,19.26533,Breë,bore
9,200189778,2023-05-09 13:26:00,#n/a,DWS-RQIS,NONE,9.19,1,#n/a,#n/a,#n/a,...,Goudini Spa Replacement Borehole BE00120 (N_GW),6,2015-03-31,2017-10-20,11,200189778,-33.66944,19.26533,Breë,bore


In [7]:
sa_wq_df = pd.DataFrame(columns=list(sorted(all_columns_set)))
for df in all_dfs:
    sa_wq_df_pre = pd.DataFrame(columns=list(sorted(all_columns_set)))

    for col in list(sa_wq_df_pre.columns):
        if col in list(df.columns):
            sa_wq_df_pre[col] = df[col]

    sa_wq_df = pd.concat([sa_wq_df, sa_wq_df_pre], axis=0, ignore_index=True)
    print(f"Length of sa_wq_df: {len(sa_wq_df)}")

Length of sa_wq_df: 29
Length of sa_wq_df: 13294
Length of sa_wq_df: 13332
Length of sa_wq_df: 13337
Length of sa_wq_df: 13539
Length of sa_wq_df: 13578
Length of sa_wq_df: 25967
Length of sa_wq_df: 26130
Length of sa_wq_df: 30144
Length of sa_wq_df: 30178
Length of sa_wq_df: 34287
Length of sa_wq_df: 35169
Length of sa_wq_df: 80878
Length of sa_wq_df: 80966
Length of sa_wq_df: 83226
Length of sa_wq_df: 83407
Length of sa_wq_df: 85099
Length of sa_wq_df: 85136
Length of sa_wq_df: 85666
Length of sa_wq_df: 85675
Length of sa_wq_df: 91157
Length of sa_wq_df: 92628
Length of sa_wq_df: 105786
Length of sa_wq_df: 106002
Length of sa_wq_df: 111963
Length of sa_wq_df: 112611
Length of sa_wq_df: 122268
Length of sa_wq_df: 122667
Length of sa_wq_df: 126951
Length of sa_wq_df: 128037
Length of sa_wq_df: 135799
Length of sa_wq_df: 135857
Length of sa_wq_df: 148077
Length of sa_wq_df: 148149
Length of sa_wq_df: 149101
Length of sa_wq_df: 149110
Length of sa_wq_df: 149735
Length of sa_wq_df: 150433

In [8]:
make_dataframe_cols_lowercase(sa_wq_df)
sa_wq_df["year"] = sa_wq_df["date_time"].str.split("-").str[0]
sa_wq_df["month"] = sa_wq_df["date_time"].str.split("-").str[1]
sa_wq_df["day"] = sa_wq_df["date_time"].str.split("-").str[2]
sa_wq_df["day"] = sa_wq_df["day"].str.split(" ").str[0]
sa_wq_df["date"] = sa_wq_df["year"] + "-" + sa_wq_df["month"] + "-" + sa_wq_df["day"]
sa_wq_df["sample date"] = pd.to_datetime(sa_wq_df["date"], format='%Y-%m-%d').dt.strftime('%d-%m-%Y')
sa_wq_df = sa_wq_df.rename(columns={'data_lat': 'latitude', 'data_lng': 'longitude'})
add_formatted_join_column(sa_wq_df, "join_column")
sa_wq_df.replace('#n/a', np.nan, inplace=True)

for col in DATA_COLUMNS:
    sa_wq_df[col] = sa_wq_df[col].astype(float)

In [9]:
sa_wq_df.to_csv("extracted_data/extracted_depWatSanData.csv", index=False)

In [36]:
from tqdm import tqdm

ordered_cols = PRIMARY_COLUMNS
ordered_cols.extend(DATA_COLUMNS)
df_all_rows = None

for index, wq_row in tqdm(wq_df.iterrows(), desc="Process Water Quality Data"):
    lat = wq_row["latitude"]
    lon = wq_row["longitude"]
    sample_date = wq_row["sample date"]
    sample_date_plus = wq_row["sample date plus"]
    sample_date_minus = wq_row["sample date minus"]

    sa_wq_df_filter = sa_wq_df[(sample_date_minus <= sa_wq_df["sample date (date)"]) & (sa_wq_df["sample date (date)"] <= sample_date_plus)]
    sa_wq_df_filter['euc_dist_dec_deg'] = np.sqrt((sa_wq_df_filter['latitude'] - lat)**2 + (sa_wq_df_filter['longitude'] - lon)**2)
    sa_wq_df_filter = sa_wq_df_filter.sort_values(by='euc_dist_dec_deg')
    sa_wq_df_filter['latitude_rad'] = np.radians(sa_wq_df_filter['latitude'])
    sa_wq_df_filter['longitude_rad'] = np.radians(sa_wq_df_filter['longitude'])

    ref_point_rad = [radians(lat), radians(lon)]

    sa_wq_df_filter['euc_dist_km'] = haversine_distances(sa_wq_df_filter[['latitude_rad', 'longitude_rad']], [ref_point_rad]) * 6371
    sa_wq_df_filter = sa_wq_df_filter.sort_values(by='euc_dist_km')
    sa_wq_df_filter = sa_wq_df_filter[sa_wq_df_filter['euc_dist_km'] < 100.0]

    sa_wq_df_filter = sa_wq_df_filter[DATA_COLUMNS]
    sa_wq_df_filter = sa_wq_df_filter.mean()

    sa_wq_df_filter['latitude'] = lat
    sa_wq_df_filter['longitude'] = lon
    sa_wq_df_filter['sample date'] = sample_date

    sa_wq_df_filter = sa_wq_df_filter[ordered_cols]

    if df_all_rows is None:
        df_all_rows = sa_wq_df_filter.to_frame().T
    else:
        df_all_rows = pd.concat([df_all_rows, sa_wq_df_filter.to_frame().T], axis=0, ignore_index=True)

Process Water Quality Data: 9319it [01:20, 116.20it/s]


In [45]:
df_all_rows_process = df_all_rows.copy()

In [42]:
df_all_rows_process

,latitude,longitude,sample date,mg_diss_water_dl,cl_diss_water_dl,ca_diss_water_dl,no3_no2_n_diss_water_dl,cl_diss_water,ec_phys_water,p_tot_water_dl,...,ec_phys_water_dl,p_tot_water,si_diss_water_dl,f_diss_water,po4_p_diss_water_dl,tal_diss_water_dl,k_diss_water_dl,kjel_n_tot_water,so4_diss_water_dl,k_diss_water
0,-28.760833,17.730278,02-01-2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-26.861111,28.884722,03-01-2011,1.392308,0.343636,0.515385,0.0655,57.945333,60.605185,0.009231,...,0.373913,0.083231,0.125,0.411455,0.075636,3.688235,0.116,0.934538,1.957692,4.759333
2,-26.45,28.085833,03-01-2011,1.458824,0.6964,1.0,0.051,50.122552,63.80675,0.0093,...,0.65,0.1228,0.125,0.278056,0.028267,5.7,0.9795,1.37835,2.961538,3.142917
3,-27.671111,27.236944,03-01-2011,1.5,0.09,1.0,2.325,12.542,111.292857,NaN,...,1.0,NaN,0.125,0.083,0.929286,5.0,2.0,NaN,3.0,3.339
4,-27.356667,27.286389,03-01-2011,1.5,0.32875,1.0,1.622,59.527778,111.62,0.012,...,0.9625,0.018,0.125,0.092,0.6508,5.0,1.461714,0.42,3.875,4.524857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9314,-27.5275,30.858056,23-12-2015,1.333333,2.0,2.333333,0.171667,18.133333,44.406286,0.02,...,1.0,0.08,0.7,0.780333,0.182258,16.666667,1.533333,0.8745,4.116667,4.0
9315,-26.861111,28.884722,23-12-2015,1.5,1.785714,2.5,0.1,26.327929,53.447143,0.02,...,1.0,0.102083,1.0,0.2635,0.02,10.0,1.0,1.47575,1.457143,6.244429
9316,-26.984722,26.632278,23-12-2015,1.5,1.8,2.5,0.1,45.6572,67.5,0.02,...,1.0,0.01,1.0,0.4186,0.02,10.0,1.0,1.2,1.92,5.3284
9317,-27.935,26.126667,23-12-2015,1.5,2.0,2.5,0.1,66.4,76.3,0.02,...,1.0,0.054,1.0,0.224,0.02,10.0,0.8,2.045,1.2,11.5


In [46]:
num_rows = df_all_rows_process.shape[0]
for col in DATA_COLUMNS:
    num_na = df_all_rows_process[col].isna().sum()
    perc_null = num_na / num_rows * 100
    print(f"{col} - {perc_null}%")
    if perc_null > 20:
        print(f"\t Dropping column")
        df_all_rows_process.drop(columns=col, inplace=True)

mg_diss_water_dl - 10.183496083270738%
cl_diss_water_dl - 10.36591909003112%
ca_diss_water_dl - 11.428264835282755%
no3_no2_n_diss_water_dl - 12.952033479987124%
cl_diss_water - 10.36591909003112%
ec_phys_water - 5.225882605429767%
p_tot_water_dl - 62.09893765425475%
	 Dropping column
ph_diss_water_dl - 5.322459491361734%
no3_no2_n_diss_water - 12.748148943019638%
nh4_n_diss_water_dl - 6.535035948063096%
dms_tot_water_dl - 100.0%
	 Dropping column
na_diss_water_dl - 14.379225238759524%
no3_no2_n_calc_water_dl - 100.0%
	 Dropping column
tal_diss_water - 14.894301963730014%
dms_tot_water - 41.09883034660371%
	 Dropping column
ph_diss_water - 4.453267517974031%
na_diss_water - 14.379225238759524%
po4_p_diss_water - 5.858997746539328%
mg_diss_water - 10.183496083270738%
nh4_n_diss_water - 6.535035948063096%
kjel_n_tot_water_dl - 60.9507457881747%
	 Dropping column
si_diss_water - 20.409915226955682%
	 Dropping column
so4_diss_water - 9.711342418714453%
f_diss_water_dl - 19.873376971778086%

In [48]:
df_all_rows_process.columns

Index(['latitude', 'longitude', 'sample date', 'mg_diss_water_dl',
       'cl_diss_water_dl', 'ca_diss_water_dl', 'no3_no2_n_diss_water_dl',
       'cl_diss_water', 'ec_phys_water', 'ph_diss_water_dl',
       'no3_no2_n_diss_water', 'nh4_n_diss_water_dl', 'na_diss_water_dl',
       'tal_diss_water', 'ph_diss_water', 'na_diss_water', 'po4_p_diss_water',
       'mg_diss_water', 'nh4_n_diss_water', 'so4_diss_water',
       'f_diss_water_dl', 'ca_diss_water', 'ec_phys_water_dl', 'f_diss_water',
       'po4_p_diss_water_dl', 'tal_diss_water_dl', 'so4_diss_water_dl'],
      dtype='str')

In [49]:
df_all_rows_process.to_csv("../data/sa_dep_water_and_sanitation/depWaterAndSan_features_training_chemReadings.csv", index=False)